# Chapter 6 – Structured Outputs
## Hands-On Colab Notebook for Network Engineers

**The lesson that nearly cost us a compliance contract:**

> *"Our AI audit tool ran nightly across 400+ routers.*
> *Week 2: it reported '0 issues found' on a router with Telnet enabled and SNMP v2c public.*
> *Root cause: the LLM changed its output format. Our text parser missed everything.*
> *A LLM is probabilistic. Your code is deterministic. Structured outputs bridge that gap."*

| # | Demo | What it teaches |
|---|------|-----------------|
| 1 | The Fragility Problem | Why text parsing breaks — 3 formats, same data |
| 2 | JSON Extraction V1→V3 | Progressive improvement: fragile → robust |
| 3 | Robust JSON Parser | Handle raw JSON, markdown blocks, embedded prose |
| 4 | Pydantic Validation | Turn raw dicts into validated Python objects |
| 5 | Tool Use Basics | Force exact schema with BGP neighbor parsing |
| 6 | Tool Use: Security Audit | Full production schema with nested objects |
| 7 | Retry with Error Feedback | Self-correcting extraction pipeline |
| 8 | Full Pipeline | Pydantic + Tool Use + retry in one pattern |

**~35 minutes** | **~$0.10 in API calls** | **No local setup required**

> Networking analogy: a structured output schema is your RFC.
> Without it, you're parsing free-form natural language descriptions of routes.
> With it, every field is at a known offset, every type is defined, missing fields throw errors.


---
## Setup

In [ ]:
!pip install anthropic pydantic -q

import json, re
from anthropic import Anthropic

try:
    from google.colab import userdata
    ANTHROPIC_API_KEY = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    import getpass
    ANTHROPIC_API_KEY = getpass.getpass("Enter your Anthropic API key: ")

client = Anthropic(api_key=ANTHROPIC_API_KEY)

HAIKU  = "claude-haiku-4-5-20251001"
SONNET = "claude-sonnet-4-20250514"

def ask(prompt, model=HAIKU, system=None, max_tokens=300, temperature=0, tools=None, tool_choice=None):
    """Minimal wrapper — temperature=0 by default for structured outputs."""
    kwargs = dict(
        model=model, max_tokens=max_tokens, temperature=temperature,
        messages=[{"role": "user", "content": prompt}]
    )
    if system:
        kwargs["system"] = system
    if tools:
        kwargs["tools"] = tools
    if tool_choice:
        kwargs["tool_choice"] = tool_choice
    return client.messages.create(**kwargs)

print("Setup complete — temperature=0 default for reliable outputs.")
print(f"Models available: {HAIKU.split('-')[1]}, {SONNET.split('-')[1]}")


---
## Demo 1 — The Fragility Problem

**This is exactly what broke our compliance tool.**

A LLM doesn't return the same format every time.
Your text parser handles Format A perfectly — but formats B and C silently return zero results.

Three real outputs from the same prompt. Same data. Different format:
- Format A: dash-prefixed list (what your parser expected)
- Format B: numbered list with severity labels
- Format C: bullet points with prose

The parser below handles only Format A. Watch it fail on B and C.


In [ ]:
# The fragile parser that nearly cost us a compliance contract
def parse_findings_v1(llm_response: str) -> list:
    """
    V1: Parse security findings from LLM response.
    Works on Format A. Silently returns [] on B and C.
    This is the bug that caused '0 issues found' on vulnerable configs.
    """
    findings = []
    for line in llm_response.split('\n'):
        if line.strip().startswith('- '):
            findings.append(line.strip()[2:])
    return findings

# Simulated LLM responses — all contain the same 3 issues
FORMAT_A = """- Telnet enabled on VTY lines
- SNMP community 'public' configured (read-only)
- No service password-encryption"""

FORMAT_B = """**Issues Found:**
1. Telnet is enabled on the VTY lines (HIGH RISK)
2. SNMP v2c public community string is active
3. Password encryption is not enabled"""

FORMAT_C = """I found the following security concerns:
• The router allows telnet connections from any source
• SNMP community strings are using legacy v2c without authentication
• Passwords are stored in cleartext in the config"""

print("Same 3 security issues, 3 different formats:")
print()
for label, response in [("A (dash list)", FORMAT_A),
                         ("B (numbered)", FORMAT_B),
                         ("C (bullets)", FORMAT_C)]:
    found = parse_findings_v1(response)
    status = "✅" if len(found) == 3 else "❌"
    print(f"  Format {label}: found {len(found)} issues {status}")

print()
print("Format B and C return 0 findings — the same silent failure that hit us in production.")
print("This is why structured outputs exist.")


---
## Demo 2 — JSON Extraction: V1 → V2 → V3

**Three versions of the same function. Each one fixes a real production failure.**

| Version | What it does | Where it fails |
|---------|-------------|----------------|
| V1 | Just call `json.loads()` | Crashes when model adds explanation text |
| V2 | Add `temperature=0` + explicit schema in prompt | Still crashes when model wraps in markdown |
| V3 | Robust parser: handles raw JSON, markdown, and embedded prose | Handles 99%+ of real cases |

We will build V3 together, step by step.


In [ ]:
# ── V1: Works in demos, crashes in production ──────────────────────────────
def extract_interfaces_v1(config_snippet: str) -> list:
    """
    V1: Ask for JSON, immediately parse it.
    Fails when the model adds 'Here is the JSON:' before the array.
    """
    response = ask(
        f"Extract interface names from this config as JSON array:\n{config_snippet}",
        max_tokens=200
    )
    # This crashes when model wraps the JSON in explanation text
    return json.loads(response.content[0].text)

# ── V2: temperature=0 + explicit schema ──────────────────────────────────────
def extract_interfaces_v2(config_snippet: str) -> list:
    """
    V2: Explicit schema in prompt + temperature=0.
    temperature=0 = same input always gives same output format.
    Still crashes if model uses markdown code blocks (```json...```).
    """
    prompt = f"""Extract interfaces from this Cisco IOS config.

Config: {config_snippet}

Return ONLY a JSON array. No other text. Exact structure:
[{{"name": "GigabitEthernet0/0", "status": "up"}}]"""

    response = client.messages.create(
        model=HAIKU,
        max_tokens=300,
        temperature=0,          # Critical: same input = same output format
        messages=[{"role": "user", "content": prompt}]
    )
    return json.loads(response.content[0].text)   # Still crashes on markdown

SAMPLE_CONFIG = """interface GigabitEthernet0/0
 ip address 10.0.0.1 255.255.255.0
 no shutdown
interface Loopback0
 ip address 172.16.0.1 255.255.255.255"""

# Test V2 — should work, shows improvement over V1
try:
    result = extract_interfaces_v2(SAMPLE_CONFIG)
    print("V2 result:", result)
except Exception as e:
    print(f"V2 failed: {e}")


In [ ]:
# ── V3: Robust parser that handles all 3 real-world formats ─────────────────
def extract_json_from_text(text: str):
    """
    Extract JSON from LLM response text.
    Handles three formats the model might return:
      1. Raw JSON (85%+ of cases)
      2. JSON in markdown code blocks ```json...```
      3. JSON embedded in prose: 'Here are the interfaces: [...]'
    """
    text = text.strip()

    # Path 1: direct parse — fastest, covers most cases
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    # Path 2: strip markdown code blocks
    for pattern in [r'```json\s*([\s\S]*?)\s*```', r'```\s*([\s\S]*?)\s*```']:
        match = re.search(pattern, text)
        if match:
            try:
                return json.loads(match.group(1).strip())
            except json.JSONDecodeError:
                continue

    # Path 3: find JSON by bracket matching
    # Walks the string finding the first [ or { and its matching closing bracket
    for open_c, close_c in [('[', ']'), ('{', '}')]:
        start = text.find(open_c)
        if start == -1:
            continue
        depth = 0
        for i, ch in enumerate(text[start:], start):
            if ch == open_c:
                depth += 1
            elif ch == close_c:
                depth -= 1
                if depth == 0:
                    try:
                        return json.loads(text[start:i+1])
                    except json.JSONDecodeError:
                        break

    return None   # All three paths failed

# Test the three format cases
FORMAT_RAW      = '[{"name": "Gi0/0", "status": "up"}]'
FORMAT_MARKDOWN = '```json\n[{"name": "Gi0/0", "status": "up"}]\n```'
FORMAT_PROSE    = 'Here are the interfaces: [{"name": "Gi0/0", "status": "up"}]'

print("Testing extract_json_from_text() on all three formats:")
for label, text in [("Raw JSON", FORMAT_RAW),
                    ("Markdown block", FORMAT_MARKDOWN),
                    ("Embedded in prose", FORMAT_PROSE)]:
    result = extract_json_from_text(text)
    status = "✅" if result else "❌"
    print(f"  {label}: {status} → {result}")


### The Parser Evolution at a Glance

| Version | What it handles | Reliability |
|---------|----------------|-------------|
| V1 | Clean JSON only | ~60% — breaks on markdown code blocks |
| V2 | Strips markdown fences | ~80% — still fails on mixed text+JSON |
| V3 | Regex extraction from any text | ~92% — handles all common LLM output formats |
| V4 (next) | Full schema prompt + V3 parser | ~95% — model knows the exact shape you need |

> **Network analogy:** V1 is like expecting only clean CDP output.
> V3 handles whatever format the device returns — like a good NMS parser.

---
## Demo 3 — V4: Production-Ready Interface Parser

**The full schema prompt is what takes you from ~90% to ~99%.**

Key ideas in V4:
- Clear schema in the prompt with exact field names and types
- `null` explicitly called out (not empty string, not "none")
- Rules section covers edge cases: `admin-down` for shutdown interfaces
- Use Sonnet for complex parsing — better at following multi-rule schemas


In [ ]:
def extract_interfaces_v4(config: str) -> list:
    """
    V4: Production-ready. Full schema, robust parsing, sanity check.
    Compare with V1 (10 lines) and V2 (20 lines) — each line adds reliability.
    """
    prompt = f"""You are a Cisco IOS configuration parser.

Extract ALL interfaces from this configuration.

Config:
{config}

Return a JSON array with this EXACT structure for each interface:
[
  {{
    "name": "full interface name (e.g. GigabitEthernet0/0)",
    "ip_address": "IP address string, or null if not configured",
    "subnet_mask": "subnet mask string, or null if not configured",
    "description": "interface description, or null if not set",
    "status": "up | down | admin-down"
  }}
]

Rules:
- Include ALL interfaces, even those without IP addresses
- Use null (not empty string) for missing optional fields
- status = admin-down if the interface has 'shutdown' configured
- Return ONLY the JSON array, starting with [ and ending with ]"""

    response = client.messages.create(
        model=SONNET,           # Sonnet for complex multi-rule schemas
        max_tokens=1000,
        temperature=0,
        messages=[{"role": "user", "content": prompt}]
    )

    raw = response.content[0].text
    extracted = extract_json_from_text(raw)

    if extracted is None:
        print(f"WARNING: Could not extract JSON. Raw response preview: {raw[:100]}")
        return []

    # Sanity check — empty result on a config that has interfaces is suspicious
    if len(extracted) == 0 and 'interface' in config.lower():
        print("WARNING: Parser returned empty list but config contains interfaces")

    return extracted

# Test on a realistic config
ROUTER_CONFIG = """hostname core-router-01
!
interface GigabitEthernet0/0
 description WAN to ISP-A
 ip address 203.0.113.1 255.255.255.252
 no shutdown
!
interface GigabitEthernet0/1
 description LAN - Server Segment
 ip address 10.10.10.1 255.255.255.0
 no shutdown
!
interface Loopback0
 description BGP Source
 ip address 172.16.255.1 255.255.255.255
!
interface GigabitEthernet0/2
 description Future Expansion
 shutdown"""

interfaces = extract_interfaces_v4(ROUTER_CONFIG)
print(f"Extracted {len(interfaces)} interfaces:\n")
for iface in interfaces:
    ip   = iface.get('ip_address') or 'no IP'
    desc = iface.get('description') or 'no description'
    stat = iface.get('status', '?')
    print(f"  [{stat:>11}]  {iface['name']:<28}  {ip:<18}  ({desc})")


---
## Demo 4 — Pydantic: Your Data Contract Enforcer

**Raw JSON gives you a dict. Pydantic gives you a validated Python object.**

Without Pydantic, `interface["status"]` might be `"up"`, `"UP"`, `"active"`, or `"online"`.
Your downstream code has to handle all of this.

With Pydantic, you declare that `status` must be one of `["up", "down", "admin-down"]`.
Anything else? `ValidationError` — caught immediately at the boundary, not silently failing deep in your pipeline 3 days later.

> Networking analogy: Pydantic is the ACL at the edge of your data pipeline.
> You define what is allowed. Everything else is rejected with a clear error message.


In [ ]:
from pydantic import BaseModel, field_validator
from typing import Optional
from enum import Enum

# ── Simple model first — understand the concept ──────────────────────────────
class NetworkDevice(BaseModel):
    """The simplest Pydantic model — three fields, all required."""
    hostname: str
    ip_address: str
    vendor: str

# Direct instantiation from a dict (what json.loads() returns)
raw_data = {"hostname": "dist-sw-02", "ip_address": "10.5.20.1", "vendor": "Cisco"}
device = NetworkDevice(**raw_data)
print(f"Valid device: {device.hostname} at {device.ip_address}")

# Wrong type? Pydantic catches it
from pydantic import ValidationError
try:
    bad = NetworkDevice(**{"hostname": 12345, "ip_address": "10.5.20.1", "vendor": "Cisco"})
except ValidationError as e:
    print(f"Validation caught bad hostname type: {e.errors()[0]['msg']}")

# Missing field? Pydantic catches it
try:
    bad = NetworkDevice(**{"hostname": "sw-01", "vendor": "Cisco"})  # missing ip_address
except ValidationError as e:
    print(f"Validation caught missing field: {e.errors()[0]['msg']}")


In [ ]:
import ipaddress

class InterfaceStatus(str, Enum):
    UP         = "up"
    DOWN       = "down"
    ADMIN_DOWN = "admin-down"

class Interface(BaseModel):
    """
    Network interface with business-logic validation.
    The Enum for status guarantees only three values ever reach your code.
    """
    name:        str
    ip_address:  Optional[str] = None
    subnet_mask: Optional[str] = None
    description: Optional[str] = None
    status:      InterfaceStatus          # Enum — only 'up', 'down', 'admin-down' allowed

    @field_validator('ip_address')
    @classmethod
    def validate_ip(cls, v):
        if v is None:
            return v
        try:
            ipaddress.IPv4Address(v)
            return v
        except ValueError:
            raise ValueError(f"Invalid IP address format: '{v}'")

    def cidr(self) -> Optional[str]:
        """Return IP in CIDR notation like 10.0.0.1/30"""
        if not self.ip_address or not self.subnet_mask:
            return None
        try:
            net = ipaddress.IPv4Network(f"0.0.0.0/{self.subnet_mask}", strict=False)
            return f"{self.ip_address}/{net.prefixlen}"
        except ValueError:
            return None

# Validate the interfaces extracted in Demo 3
print("Validating extracted interfaces through Pydantic:\n")
validated = []
errors = 0
for raw in interfaces:
    try:
        iface = Interface(**raw)
        validated.append(iface)
    except ValidationError as e:
        errors += 1
        print(f"  Validation failed for {raw.get('name','?')}: {e.errors()[0]['msg']}")

print(f"  Result: {len(validated)} valid / {errors} failed\n")
for iface in validated:
    cidr = iface.cidr() or "no IP"
    print(f"  {iface.name:<30} {str(iface.status.value):<12} {cidr}")

print()
print("status is now an Enum — iface.status == InterfaceStatus.UP works perfectly.")
print("No more 'UP' vs 'up' vs 'active' string comparison bugs.")


---
## Demo 5 — Tool Use: Force Exact Schema

**This is the biggest leap in reliability: from ~90% to ~99%+.**

With prompt-enforced JSON, you are *asking* the model to use a format.
With Tool Use, you are *forcing* it — the API itself enforces the schema.

The concept is simple:
1. Define a tool with a JSON schema (like a function signature)
2. Tell the model: "use this tool to give your answer"
3. The model returns structured JSON matching your schema exactly
4. You extract it from `block.input` — already a Python dict, no parsing needed

> Networking analogy: Tool Use is BGP attribute-based filtering.
> Instead of hoping the model chooses the right format,
> you define explicit attributes it must fill. The schema is enforced, not requested.


In [ ]:
# The simplest possible Tool Use — BGP neighbor parsing
# One tool, three fields, no nested objects

BGP_TOOL = {
    "name": "parse_bgp_neighbor",
    "description": "Extract BGP neighbor details from CLI output into structured JSON.",
    "input_schema": {
        "type": "object",
        "properties": {
            "neighbor_ip": {
                "type": "string",
                "description": "The IP address of the BGP neighbor"
            },
            "remote_asn": {
                "type": "integer",
                "description": "The remote Autonomous System Number"
            },
            "state": {
                "type": "string",
                "enum": ["Established", "Active", "Idle", "Connect", "OpenSent", "OpenConfirm"],
                "description": "Current BGP session state"
            },
            "uptime": {
                "type": "string",
                "description": "How long the session has been up, or null if not Established"
            },
            "prefixes_received": {
                "type": "integer",
                "description": "Number of prefixes received from this neighbor"
            }
        },
        "required": ["neighbor_ip", "remote_asn", "state"]
    }
}

# Messy unstructured BGP output (what you get from 'show bgp summary')
RAW_BGP = """
BGP neighbor is 10.0.0.5, remote AS 65001, internal link
  BGP state = Established, up for 1d02h30m
  Neighbor capabilities: Route refresh: advertised and received
  Prefixes received: 4832
  Last read 00:00:23, last write 00:00:18, hold time is 90
"""

response = client.messages.create(
    model=HAIKU,
    max_tokens=300,
    tools=[BGP_TOOL],
    tool_choice={"type": "tool", "name": "parse_bgp_neighbor"},  # Force this specific tool
    messages=[{"role": "user", "content": f"Parse this BGP output:\n{RAW_BGP}"}]
)

# Extract the result — always a dict matching the schema, no JSON parsing needed
bgp_data = None
for block in response.content:
    if block.type == "tool_use":
        bgp_data = block.input
        break

print("Tool Use result (already a Python dict — no json.loads() needed):")
print(f"  Neighbor: {bgp_data['neighbor_ip']}")
print(f"  Remote AS: {bgp_data['remote_asn']}")
print(f"  State: {bgp_data['state']}")
print(f"  Uptime: {bgp_data.get('uptime', 'N/A')}")
print(f"  Prefixes: {bgp_data.get('prefixes_received', 'N/A')}")
print()
print("Compare: prompt-enforced JSON → json.loads(text) — can crash")
print("         Tool Use            → block.input        — always a dict")


### Tool Use vs Text Parsing: The Tradeoff

| Method | Reliability | Speed | Cost | Best for |
|--------|------------|-------|------|----------|
| Text + regex parsing | ~85% | Fastest | Cheapest | Quick scripts, prototyping |
| Text + Pydantic validation | ~95% | Fast | Cheap | Data validation, type checking |
| Tool Use (structured output) | ~99% | Slightly slower | ~10% more | Production pipelines |

> **When to use which:** Start with text parsing for prototypes.
> Move to Tool Use when reliability matters (compliance tools, ticket automation).
> The 10% cost increase is negligible compared to fixing bad data downstream.

---
## Demo 6 — Tool Use: Full Security Audit Schema

**Nested objects, enums, integer constraints — Tool Use handles all of it.**

This is the audit tool that nearly failed us in Demo 1.
But now with Tool Use instead of text parsing:
- `overall_risk_score` is always an integer between 0 and 100
- Each finding always has `title`, `description`, and `remediation`
- `platform` is always one of six valid values
- No format variation possible — schema is enforced by the API


In [ ]:
SECURITY_AUDIT_TOOL = {
    "name": "report_security_findings",
    "description": "Report all security findings from a network device configuration audit.",
    "input_schema": {
        "type": "object",
        "properties": {
            "device_hostname": {"type": "string"},
            "platform": {
                "type": "string",
                "enum": ["IOS", "IOS-XE", "NX-OS", "Junos", "FortiOS", "unknown"]
            },
            "critical_findings": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "title":       {"type": "string"},
                        "description": {"type": "string"},
                        "config_line": {"type": "string"},
                        "remediation": {"type": "string"}
                    },
                    "required": ["title", "description", "remediation"]
                }
            },
            "high_findings": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "title":       {"type": "string"},
                        "description": {"type": "string"},
                        "remediation": {"type": "string"}
                    },
                    "required": ["title", "remediation"]
                }
            },
            "passed_checks":     {"type": "array", "items": {"type": "string"}},
            "overall_risk_score": {
                "type": "integer", "minimum": 0, "maximum": 100,
                "description": "0 = no risk, 100 = maximum risk"
            },
            "audit_summary": {"type": "string"}
        },
        "required": [
            "device_hostname", "platform", "critical_findings",
            "high_findings", "passed_checks", "overall_risk_score", "audit_summary"
        ]
    }
}

VULNERABLE_CONFIG = """hostname branch-fw-01
!
enable password cisco
!
line vty 0 4
 transport input all
 password letmein
!
snmp-server community public RO
snmp-server community private RW
!
no service password-encryption
no logging
!
interface GigabitEthernet0/0
 ip address 192.168.1.1 255.255.255.0
 no shutdown"""

print("Running security audit via Tool Use...")
response = client.messages.create(
    model=SONNET,
    max_tokens=1500,
    temperature=0,
    tools=[SECURITY_AUDIT_TOOL],
    tool_choice={"type": "any"},    # Force model to use a tool — cannot return plain text
    system="You are a network security auditor. Analyze the config for vulnerabilities.",
    messages=[{"role": "user", "content": f"Audit this config:\n\n{VULNERABLE_CONFIG}"}]
)

findings = None
for block in response.content:
    if block.type == "tool_use":
        findings = block.input
        break

print(f"\nDevice:     {findings['device_hostname']}")
print(f"Platform:   {findings['platform']}")
print(f"Risk Score: {findings['overall_risk_score']}/100")
print(f"\nCritical Issues ({len(findings['critical_findings'])}):")
for f in findings['critical_findings']:
    print(f"  [{f['title']}]")
    print(f"    Fix: {f['remediation']}")
print(f"\nHigh Issues ({len(findings['high_findings'])}):")
for f in findings['high_findings']:
    print(f"  [{f['title']}]: {f['remediation']}")
print(f"\nPassed Checks ({len(findings['passed_checks'])}):")
for c in findings['passed_checks']:
    print(f"  ✓ {c}")
print(f"\nSummary: {findings['audit_summary']}")


---
## Demo 7 — Retry with Error Feedback

**Naive retry repeats the same mistake. Smart retry shows the model what went wrong.**

Even with Tool Use, edge cases happen:
- Config is too large → token limit hit → partial output
- Schema validation passes but logic is wrong (risk score 5 with 3 critical findings?)

The pattern: on failure, send the error back to the model as context.
The model sees its own mistake and corrects it.

> Networking analogy: this is like OSPF graceful restart.
> The process recovers using the last known good state as context.


In [ ]:
import time

def validate_audit_logic(findings: dict) -> None:
    """
    Business logic validation beyond schema validation.
    Schema ensures structure. This ensures logical consistency.
    """
    critical_count = len(findings.get('critical_findings', []))
    risk_score     = findings.get('overall_risk_score', 0)

    # A config with critical findings can't have a low risk score
    if critical_count > 0 and risk_score < 30:
        raise ValueError(
            f"Logic error: {critical_count} critical findings but risk score = {risk_score}. "
            "Risk score must be >= 30 when critical issues exist."
        )

    # Every high/critical finding must have a remediation step
    for severity in ['critical_findings', 'high_findings']:
        for f in findings.get(severity, []):
            if not f.get('remediation', '').strip():
                raise ValueError(f"Finding '{f.get('title','?')}' has empty remediation.")

def audit_with_retry(config: str, max_retries: int = 3) -> dict:
    """
    Run security audit with retry and error feedback.
    Attempt 1: clean call.
    Attempts 2+: include the previous error so the model can self-correct.
    """
    last_error = None
    last_raw   = None

    for attempt in range(1, max_retries + 1):
        try:
            if attempt == 1:
                # First attempt — clean
                messages = [
                    {"role": "user",
                     "content": f"Audit this network config:\n\n{config}"}
                ]
            else:
                # Include error context — the model sees what went wrong
                messages = [
                    {"role": "user",
                     "content": f"Audit this network config:\n\n{config}"},
                    {"role": "assistant",
                     "content": f"[Previous attempt output: {last_raw}]"},
                    {"role": "user",
                     "content": (
                         f"Your previous attempt failed validation:\n{last_error}\n\n"
                         "Please correct the issue and call the tool again."
                     )}
                ]

            response = client.messages.create(
                model=SONNET, max_tokens=1500, temperature=0,
                tools=[SECURITY_AUDIT_TOOL],
                tool_choice={"type": "any"},
                system="You are a network security auditor.",
                messages=messages
            )

            result = None
            for block in response.content:
                if block.type == "tool_use":
                    result = block.input
                    last_raw = str(result)
                    break

            if result is None:
                raise ValueError("Model did not call the security audit tool.")

            validate_audit_logic(result)   # Business logic check
            print(f"  Attempt {attempt}: ✅ passed validation")
            return result

        except Exception as e:
            last_error = str(e)
            print(f"  Attempt {attempt}: ❌ {e}")
            if attempt < max_retries:
                wait = 2 ** attempt
                print(f"  Retrying in {wait}s...")
                time.sleep(wait)

    print(f"All {max_retries} attempts failed.")
    return None

print("Testing retry pattern on the vulnerable config:")
result = audit_with_retry(VULNERABLE_CONFIG)
if result:
    print(f"\nFinal result: risk score {result['overall_risk_score']}/100, "
          f"{len(result['critical_findings'])} critical findings")


---
## Demo 8 — Full Pipeline: Tool Use + Pydantic + Retry

**The production pattern. All three layers combined.**

```
Config text
    │
    ▼
Tool Use ────────────────── enforces JSON schema (API-level guarantee)
    │
    ▼
Pydantic ────────────────── validates types, enums, IP format (Python-level guarantee)
    │
    ▼
Business logic check ─────── ensures logical consistency (domain-level guarantee)
    │
    ▼
Retry with error feedback ── self-corrects if any layer fails
```

Three validation layers. If all three pass, the data is trustworthy.
If any fails, the model sees the error and corrects before returning to your code.


In [ ]:
from pydantic import BaseModel
from typing import List, Optional

# ── Pydantic model for the Tool Use output ───────────────────────────────────
class SecurityFinding(BaseModel):
    title:       str
    description: str
    remediation: str
    config_line: Optional[str] = None

class AuditReport(BaseModel):
    device_hostname:    str
    platform:           str
    critical_findings:  List[SecurityFinding]
    high_findings:      List[dict]          # Simplified for this demo
    passed_checks:      List[str]
    overall_risk_score: int
    audit_summary:      str

    @property
    def risk_level(self) -> str:
        if self.overall_risk_score >= 70:
            return "CRITICAL"
        elif self.overall_risk_score >= 40:
            return "HIGH"
        elif self.overall_risk_score >= 20:
            return "MEDIUM"
        return "LOW"

def run_full_audit(config: str) -> Optional[AuditReport]:
    """
    Full pipeline: Tool Use -> Pydantic -> business logic -> retry.
    Returns a validated AuditReport object or None on failure.
    """
    raw = audit_with_retry(config)  # Tool Use + retry (from Demo 7)
    if raw is None:
        return None

    try:
        report = AuditReport(**raw)  # Pydantic validation
        return report
    except Exception as e:
        print(f"Pydantic validation failed: {e}")
        return None

# ── Run on three different configs ───────────────────────────────────────────
configs = {
    "branch-fw-01 (bad)":  VULNERABLE_CONFIG,
    "core-router-01 (OK)": ROUTER_CONFIG,
}

print("Full audit pipeline — running on multiple devices:")
print("=" * 60)
for device_label, config in configs.items():
    print(f"\n>> {device_label}")
    report = run_full_audit(config)
    if report:
        print(f"   Risk:     {report.risk_level} ({report.overall_risk_score}/100)")
        print(f"   Critical: {len(report.critical_findings)} findings")
        print(f"   Platform: {report.platform}")
        if report.critical_findings:
            print(f"   Top finding: {report.critical_findings[0].title}")
            print(f"   Fix: {report.critical_findings[0].remediation}")
    else:
        print("   Audit failed — check logs above")


---
## Summary

**The structured outputs progression in one diagram:**

```
Prototype ──────────────────────────────────────────────────── Production
    │                                                               │
    V1                V2              V3              Tool Use + Pydantic
json.loads()    temp=0 + schema   robust parser    schema enforced by API
    │                │                │                    │
crashes on      crashes on        handles 99%          always matches
explanation     markdown           of formats           your schema
text            blocks                                 exactly
```

**The three rules you now know:**
1. Always use `temperature=0` for structured outputs
2. Always use `extract_json_from_text()` not `json.loads()` directly
3. For production: use Tool Use (`tool_choice`) — the API enforces the schema

**The three validation layers for production:**
1. Tool Use schema — enforced by the API (structure guarantee)
2. Pydantic model — enforced at your code boundary (type/enum guarantee)
3. Business logic validators — domain rules the schema can't express (semantic guarantee)

> Remember the compliance contract story.
> The difference between "works usually" and "is reliable" is structured outputs.
